# Stage 1 — Wikipedia Fact-Check v2 (RAG) — Colab Run

**Resumes from a 215-row checkpoint.** Remaining: ~785 samples. Expected runtime: ~25 min.

Run cells top-to-bottom in order. Do not skip cells.

In [ ]:
# Cell 1 — Install dependencies
# Run once per Colab session. Takes ~2 minutes.
!pip install -q \
    torch torchvision \
    "transformers>=4.38" \
    accelerate \
    sentence-transformers \
    spacy \
    scikit-learn \
    pandas \
    requests \
    numpy
print("Dependencies installed.")

In [ ]:
# Cell 2 — Download spaCy model (en_core_web_lg, ~560 MB)
# Run once per Colab session.
!python -m spacy download en_core_web_lg
print("spaCy model ready.")

In [ ]:
# Cell 3 — Upload handoff files
# Upload ALL of the following files from the colab_stage1_handoff/ directory:
#   wikipedia_factcheck.py
#   wikipedia_factcheck_v2.py
#   path_overrides.py
#   run_colab.py
#   mmfakebench_factcheck_checkpoint_v2.csv
#   mmfakebench_factcheck_scores.csv
#
# Then upload MMFakeBench_val.json separately in Cell 4 (needs its own subdirectory).

from google.colab import files
print("Select the 6 files listed above (Ctrl/Cmd+click to multi-select):")
uploaded = files.upload()
print(f"Uploaded: {list(uploaded.keys())}")

In [ ]:
# Cell 4 — Upload dataset JSON into the required subdirectory
import os, shutil
from google.colab import files

os.makedirs("/content/dataset/MMFakeBench", exist_ok=True)

print("Select MMFakeBench_val.json:")
uploaded_ds = files.upload()

# Move the uploaded file to the correct location
for fname in uploaded_ds:
    src = f"/content/{fname}"
    dst = f"/content/dataset/MMFakeBench/{fname}"
    shutil.move(src, dst)
    print(f"Moved {fname} -> {dst}")

In [ ]:
# Cell 5 — Verify file structure
# All required files must show 'OK' before proceeding.
import os

required = [
    "/content/wikipedia_factcheck.py",
    "/content/wikipedia_factcheck_v2.py",
    "/content/path_overrides.py",
    "/content/run_colab.py",
    "/content/mmfakebench_factcheck_checkpoint_v2.csv",
    "/content/mmfakebench_factcheck_scores.csv",
    "/content/dataset/MMFakeBench/MMFakeBench_val.json",
]

all_ok = True
for path in required:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status}]  {path}  ({size:,} bytes)")
    if not exists:
        all_ok = False

print()
if all_ok:
    print("All files present. Ready to proceed.")
else:
    print("ERROR: Missing files — re-upload before continuing.")

In [ ]:
# Cell 6 — Verify Wikipedia connectivity
# Must return True before launching the evaluation.
import sys
sys.path.insert(0, "/content")

import path_overrides  # patches all Windows paths to /content/
from wikipedia_factcheck_v2 import check_wikipedia_connectivity

result = check_wikipedia_connectivity()
print("Wikipedia connectivity:", result)

if not result:
    print("STOP: Wikipedia is unreachable. Wait a few minutes and re-run this cell.")
    print("Do NOT proceed to Cell 7 until this returns True.")
else:
    print("Connectivity confirmed. Proceed to Cell 7.")

In [ ]:
# Cell 7 — Main evaluation run
# Automatically resumes from the 215-row checkpoint.
# Circuit breaker will handle transient WiFi drops (waits up to 5 min, saves checkpoint, exits).
# If it exits early due to a network drop, re-run this cell — resume is automatic.
# Expected runtime: ~25 minutes for the remaining ~785 samples.
!python /content/run_colab.py

In [ ]:
# Cell 8 — V1 vs V2 comparison summary
# Run only after Cell 7 completes successfully (exit code 0).
# Prints the full comparison table: coverage, factcheck_score stats, per-class breakdown.
!python /content/run_colab.py --summary-only

In [ ]:
# Cell 9 — Download output CSVs
# Downloads both the final scores file and the checkpoint (same content, belt-and-suspenders).
import os
from google.colab import files

outputs = [
    "/content/mmfakebench_factcheck_scores_v2.csv",
    "/content/mmfakebench_factcheck_checkpoint_v2.csv",
]

for path in outputs:
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"Downloading {path} ({size:,} bytes) ...")
        files.download(path)
    else:
        print(f"WARNING: {path} not found — evaluation may not have completed.")